# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [3]:
# Load the dataset
df = pd.read_csv(
    'AviationData.csv', 
    encoding='cp1252',
    low_memory=False
)
df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [10]:
# Display no. of rows and columns in the dataset
print(f"Shape of the dataset: {df.shape}")

Shape of the dataset: (88889, 31)


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

In [12]:
# Check for missing values in the dataset
df.isna().sum().sort_values(ascending=False)

Schedule                  76307
Air.carrier               72241
FAR.Description           56866
Aircraft.Category         56602
Longitude                 54516
Latitude                  54507
Airport.Code              38757
Airport.Name              36185
Broad.phase.of.flight     27165
Publication.Date          13771
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Fatal.Injuries      11401
Engine.Type                7096
Report.Status              6384
Purpose.of.flight          6192
Number.of.Engines          6084
Total.Uninjured            5912
Weather.Condition          4492
Aircraft.damage            3194
Registration.Number        1382
Injury.Severity            1000
Country                     226
Amateur.Built               102
Model                        92
Make                         63
Location                     52
Accident.Number               0
Investigation.Type            0
Event.Id                      0
Event.Date                    0
dtype: i

In [13]:
# Display summary statistics
print(df.describe(include='all'))

              Event.Id Investigation.Type Accident.Number  Event.Date  \
count            88889              88889           88889       88889   
unique           87951                  2           88863       14782   
top     20001214X45071           Accident      ERA22LA103  1982-05-16   
freq                 3              85015               2          25   
mean               NaN                NaN             NaN         NaN   
std                NaN                NaN             NaN         NaN   
min                NaN                NaN             NaN         NaN   
25%                NaN                NaN             NaN         NaN   
50%                NaN                NaN             NaN         NaN   
75%                NaN                NaN             NaN         NaN   
max                NaN                NaN             NaN         NaN   

             Location        Country Latitude Longitude Airport.Code  \
count           88837          88663    34382     3

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [14]:
# Check the unique aircraft categories
df["Aircraft.Category"].value_counts()

Aircraft.Category
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64

In [15]:
# Check whether aircraft is amateur built or not
df["Amateur.Built"].value_counts()

Amateur.Built
No     80312
Yes     8475
Name: count, dtype: int64

In [16]:
# Display the first five event dates
df["Event.Date"].head()

0    1948-10-24
1    1962-07-19
2    1974-08-30
3    1977-06-19
4    1979-08-02
Name: Event.Date, dtype: str

In [17]:
# Check missing values in specific columns
df[["Aircraft.Category", "Amateur.Built", "Event.Date"]].isna().sum()

Aircraft.Category    56602
Amateur.Built          102
Event.Date               0
dtype: int64

In [18]:
# Convert event dates to datetime format
df["Event.Date"] = pd.to_datetime(df["Event.Date"], errors='coerce')

# Calculate the cutoff date for the last 40 years
cutoff_date = (pd.Timestamp.today().normalize() - pd.DateOffset(years=40))

cutoff_date

Timestamp('1986-10-23 00:00:00')

In [19]:
# Keep events from the last 40 years
df = df[df["Event.Date"] >= cutoff_date]

# Keep only airplane accidents
df = df[df["Aircraft.Category"] == "Airplane"]

# Keep only professionally built aircraft
df = df[df["Amateur.Built"] == "No"]

In [20]:
# Confirm the dataset after filtering
df.shape

(21416, 31)

In [21]:
# Confirm the remaining aircraft categories
df["Aircraft.Category"].value_counts()

Aircraft.Category
Airplane    21416
Name: count, dtype: int64

In [22]:
# Confirm the dates after filtering
df["Event.Date"].head()

16646   1987-01-15
16647   1987-01-15
16837   1987-02-14
17151   1987-04-02
19663   1988-02-16
Name: Event.Date, dtype: datetime64[us]

In [23]:
# Confirm the remaining build types
df["Amateur.Built"].value_counts()

Amateur.Built
No    21416
Name: count, dtype: int64

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [25]:
#Identify columns related to injuries
injury_columns = [
    'Total.Fatal.Injuries', 
    'Total.Serious.Injuries',
    'Total.Minor.Injuries', 
    'Total.Uninjured'
]

In [26]:
# Check missing values in injury-related columns
df[injury_columns].isna().sum()

Total.Fatal.Injuries      2732
Total.Serious.Injuries    2803
Total.Minor.Injuries      2517
Total.Uninjured            699
dtype: int64

In [27]:
# Estimate the total occupants only when all four injury counts are available
df['Total.Occupants'] = df[injury_columns].sum(axis=1, min_count=4)

# Calculate the total number of fatal and serious injuries
df['Fatal.Serious.Injuries'] = df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']

# Calculate the fraction of fatal and serious injuries relative to total occupants
df['Fatal.Serious.Fraction'] = df['Fatal.Serious.Injuries'] / df['Total.Occupants']

# Leave the fraction as NaN if total occupants is zero or missing
df['Fatal.Serious.Fraction'] = df['Fatal.Serious.Fraction'].where(df['Total.Occupants'] != 0)

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [28]:
# Check aircraft damage categories
df["Aircraft.damage"].value_counts(dropna=False)

Aircraft.damage
Substantial    16981
Destroyed       2303
NaN             1223
Minor            812
Unknown           97
Name: count, dtype: int64

In [29]:
# Remove extra spaces and standardize capitalization
df["Aircraft.damage"] = df["Aircraft.damage"].astype('string').str.strip().str.title()

In [30]:
# Map damage categories to binary values
damage_mapping = {
    "Destroyed": 1,
    "Substantial": 0,
    "Minor": 0,
}

In [31]:
# Create the destruction metric. 
# Unknown or missing values will remain NaN
df["Aircraft.Destroyed"] = df["Aircraft.damage"].map(damage_mapping)

In [32]:
# Check the destruction counts
df['Aircraft.Destroyed'].value_counts(dropna=False)

Aircraft.Destroyed
0.0    17793
1.0     2303
NaN     1320
Name: count, dtype: int64

In [33]:
# Calculate the destruction rate 
destruction_rate = df['Aircraft.Destroyed'].mean() * 100

print(f'Aircraft destruction rate: {destruction_rate:.2f}%')

Aircraft destruction rate: 11.46%


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [34]:
# Remove extra spaces and standardise capitalisation
df['Make'] = df['Make'].str.strip().str.upper()

# Replace empty strings with missing values
df['Make'] = df['Make'].replace('', np.nan)

# Remove records with missing manufacturer names
df = df.dropna(subset=['Make']).copy()

# Keep manufacturers with at least 50 records
make_counts = df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(valid_makes)].copy()

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [8]:
# Check the unique aircraft categories
df["Aircraft.Category"].value_counts()

Aircraft.Category
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64

In [36]:
model_counts

Model
172            769
737            403
152            314
182            304
172S           276
              ... 
T206G            1
BD-700-2A12      1
737-8            1
P206D            1
PA-44            1
Name: count, Length: 2024, dtype: int64

In [37]:
make_model_counts

Make            Model  
AERO COMMANDER  100        12
                100 180     1
                100-180     3
                112         2
                112A        1
                           ..
TAYLORCRAFT     DCO-65      7
                F-19        2
                F19         4
                F21B        1
                L 2M        1
Length: 2152, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [38]:
# Inspect the columns
columns_to_inspect = [
    'Engine.Type',
    'Weather.Condition',
    'Number.of.Engines',
    'Purpose.of.flight',
    'Broad.phase.of.flight'
]

for column in columns_to_inspect:
    print(f'\n{column}')
    print(df[column].value_counts(dropna=False))


Engine.Type
Engine.Type
Reciprocating      12820
NaN                 3214
Turbo Prop           927
Turbo Fan            699
Unknown              105
Turbo Jet             70
Geared Turbofan       12
Turbo Shaft            9
UNK                    1
Name: count, dtype: int64

Weather.Condition
Weather.Condition
VMC    14274
NaN     2417
IMC      904
Unk      186
UNK       76
Name: count, dtype: int64

Number.of.Engines
Number.of.Engines
1.0    13209
2.0     2463
NaN     2089
4.0       66
3.0       25
0.0        5
Name: count, dtype: int64

Purpose.of.flight
Purpose.of.flight
Personal                     9837
NaN                          3046
Instructional                2408
Aerial Application            721
Business                      408
Unknown                       301
Positioning                   268
Skydiving                     153
Aerial Observation            147
Other Work Use                121
Banner Tow                     86
Flight Test                    73
Ferry     

In [39]:
# Clean categorical columns
# Standardise categorical text
categorical_columns = [
    'Engine.Type',
    'Weather.Condition',
    'Purpose.of.flight',
    'Broad.phase.of.flight'
]

for column in categorical_columns:
    df[column] = df[column].str.strip().str.upper()
    df[column] = df[column].replace('', np.nan)

# Standardise unknown categories
unknown_mapping = {
    'UNKNOWN': np.nan,
    'UNK': np.nan
}

for column in categorical_columns:
    df[column] = df[column].replace(unknown_mapping)

# Combine duplicate purpose-of-flight labels
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace({
    'AIR RACE/SHOW': 'AIR RACE SHOW'
})

# Convert engine counts to numeric
df['Number.of.Engines'] = pd.to_numeric(
    df['Number.of.Engines'], errors='coerce'
)

# Treat zero or negative engine counts as invalid
df.loc[df['Number.of.Engines'] <= 0, 'Number.of.Engines'] = np.nan

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [40]:
# Inspect the dataframe
df.info()

<class 'pandas.DataFrame'>
Index: 17857 entries, 16646 to 88886
Data columns (total 36 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                17857 non-null  str           
 1   Investigation.Type      17857 non-null  str           
 2   Accident.Number         17857 non-null  str           
 3   Event.Date              17857 non-null  datetime64[us]
 4   Location                17853 non-null  str           
 5   Country                 17856 non-null  str           
 6   Latitude                15981 non-null  str           
 7   Longitude               15978 non-null  str           
 8   Airport.Code            11637 non-null  str           
 9   Airport.Name            11741 non-null  str           
 10  Injury.Severity         17140 non-null  str           
 11  Aircraft.damage         16795 non-null  string        
 12  Aircraft.Category       17857 non-null  str           
 13

In [41]:
# Inspect missing values
df.isna().sum().sort_values(ascending=False)

Schedule                  15724
Broad.phase.of.flight     15438
Air.carrier                9411
Airport.Code               6220
Airport.Name               6116
Report.Status              3785
Purpose.of.flight          3347
Fatal.Serious.Fraction     3338
Engine.Type                3320
Weather.Condition          2679
Total.Occupants            2598
Fatal.Serious.Injuries     2562
Total.Serious.Injuries     2440
Total.Fatal.Injuries       2374
Total.Minor.Injuries       2205
Number.of.Engines          2094
Longitude                  1879
Latitude                   1876
Aircraft.Destroyed         1144
Aircraft.damage            1062
Publication.Date            788
Injury.Severity             717
Total.Uninjured             576
FAR.Description             344
Registration.Number         164
Location                      4
Country                       1
Event.Id                      0
Event.Date                    0
Accident.Number               0
Investigation.Type            0
Model   

In [42]:
# Remove Columns
missing_percent = df.isna().mean() * 100

columns_to_drop = missing_percent[
    (missing_percent > 80) &
    (missing_percent.index != 'Broad.phase.of.flight')
].index

df = df.drop(columns=columns_to_drop).copy()

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [43]:
# Save the cleaned dataframe to a CSV file for further analysis
df.to_csv('cleaned_aviation_data.csv', index=False)

In [44]:
# Check that the cleaned CSV file exists
from pathlib import Path

Path('cleaned_aviation_data.csv').exists()

True